# 01 · Setup and verify

Install OpenFold3, download the weights, and run a quick prediction to confirm the environment works.

**Script equivalent:** `scripts/verify_setup.sh` (for local pixi installs).

> **Runs on any GPU notebook platform** — Google Colab, Kaggle, Paperspace Gradient, SageMaker Studio Lab, Lightning AI. Make sure a **GPU runtime** is selected before running.

> Each notebook mirrors a shell script in `scripts/`; you can use either form.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU! Enable a GPU runtime.'

## Install OpenFold3 + weights
Uses the pip install path (simplest across cloud platforms). The `setup_openfold` step is interactive, so we feed answers non-interactively: default cache, default dir, `1` (default checkpoint), `no` (skip the heavy integration tests).

In [ ]:
!pip install -q "openfold3[cuequivariance]"
!printf '\n\n1\nno\n' | setup_openfold

## Smoke test
Predict a tiny protein (ubiquitin); templates off, ColabFold MSA.

In [ ]:
import json, glob
seq = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
q = {"queries": {"ubiquitin": {"chains": [{"molecule_type": "protein", "chain_ids": ["A"], "sequence": seq}]}}}
open("smoke.json", "w").write(json.dumps(q))

In [ ]:
!run_openfold predict --query-json smoke.json --use-msa-server True --use-templates False --num-diffusion-samples 1 --num-model-seeds 1 --output-dir smoke_out

## Confirm it worked
`run_openfold` exits 0 even when a query fails — so check a `.cif` was actually written.

In [ ]:
import glob
cifs = glob.glob("smoke_out/**/*.cif", recursive=True)
assert cifs, "Setup check FAILED — see smoke_out/**/logs/predict_err_rank0.log"
print("Setup OK — wrote", cifs[0])